# AutoAlpha: 遗传编程挖掘 Alpha 因子

## 论文参考

- **标题**: AutoAlpha: an Efficient Hierarchical Evolutionary Algorithm for Mining Alpha Factors
- **作者**: T. Zhang
- **年份**: 2020

### 摘要

本文提出了一种基于遗传编程 (Genetic Programming, GP) 的层次化进化算法，用于自动挖掘 Alpha 因子表达式。
通过定义运算符集合 (+, -, *, /, rank, ts_mean, ts_std, delay) 和原语集 (close, open, high, low, volume)，
算法以 IC (Information Coefficient) 为适应度函数，自动搜索高预测力的因子表达式。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 遗传编程 (Genetic Programming)

GP 是一种进化计算方法，将候选因子表示为**表达式树**：

**1. 表达式树表示**

每个 Alpha 因子是一棵树，内部节点为运算符，叶节点为原语变量或常数：

$$\alpha = \text{ts\_mean}(\text{rank}(\text{close} - \text{delay}(\text{close}, 5)), 10)$$

**2. 进化操作**

- **选择 (Selection)**: 锦标赛选择，按适应度 IC 排序
- **交叉 (Crossover)**: 随机交换两棵树的子树
- **变异 (Mutation)**: 随机替换子树为新生成的随机子树

**3. 适应度函数: IC (Information Coefficient)**

$$IC = \text{corr}(\alpha_t, r_{t+1})$$

即因子值与下期收益率的 Spearman 秩相关系数。IC > 0.03 通常被认为有意义。

**4. 层次进化策略**

- 第一层: 进化简单表达式 (深度 <= 3)
- 第二层: 将优秀简单表达式作为子树，组合成复杂表达式
- 本实现简化为单层 GP: population=50, generations=30, max_depth=5

In [ ]:
# ============================================================
# Cell 3: 动画展示 - 表达式树在代际间的生长与变异
# ============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

# 模拟GP进化过程数据
n_generations = 30
pop_size = 50

# 模拟每代的最佳IC、平均IC、表达式深度
best_ics = [0.005]
avg_ics = [0.001]
avg_depths = [2.0]
best_exprs = ['close']

for g in range(1, n_generations):
    improvement = np.random.exponential(0.003) * (1 - g / n_generations)
    best_ics.append(min(best_ics[-1] + improvement, 0.08))
    avg_ics.append(best_ics[-1] * np.random.uniform(0.3, 0.7))
    avg_depths.append(min(2.0 + g * 0.1 + np.random.randn() * 0.2, 5.0))

generations = list(range(n_generations))

# 创建动画帧: 每帧显示进化到当前代的IC曲线
frames = []
for step in range(1, n_generations + 1):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=generations[:step], y=best_ics[:step],
                       mode='lines+markers', name='最佳 IC',
                       line=dict(color='#F44336', width=3),
                       marker=dict(size=5)),
            go.Scatter(x=generations[:step], y=avg_ics[:step],
                       mode='lines', name='平均 IC',
                       line=dict(color='#2196F3', width=2, dash='dash')),
            go.Bar(x=generations[:step], y=avg_depths[:step],
                   name='平均树深度', marker_color='#4CAF50', opacity=0.3,
                   yaxis='y2'),
        ],
        name=f'Gen {step-1}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='GP 进化过程: 表达式树生长与 IC 提升'),
        xaxis_title='代数 (Generation)',
        yaxis=dict(title='IC 值', range=[0, 0.1]),
        yaxis2=dict(title='平均树深度', overlaying='y', side='right', range=[0, 6]),
        height=500, width=900,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 200}, 'transition': {'duration': 100}}]),
            dict(label='\u23f8 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0,
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import os
import copy
import random
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_stock_daily, get_index_daily
from shared.backtest_engine import Backtester
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_returns_distribution
)
from shared.constants import DEFAULT_START, DEFAULT_END, ETF_CSI300

print('所有模块导入成功')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
from shared.data_fetcher import get_etf_daily

try:
    df = get_etf_daily(ETF_CSI300, DEFAULT_START, DEFAULT_END)
    print(f'ETF 510300 数据: {len(df)} 条')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    n = len(dates)
    np.random.seed(42)
    price = 4.5 + np.cumsum(np.random.randn(n) * 0.02)
    price = np.maximum(price, 2.0)
    df = pd.DataFrame({
        'open': price * (1 + np.random.randn(n) * 0.003),
        'close': price,
        'high': price * (1 + np.abs(np.random.randn(n) * 0.008)),
        'low': price * (1 - np.abs(np.random.randn(n) * 0.008)),
        'volume': np.random.randint(50000, 2000000, n).astype(float),
    }, index=dates)

print(f'数据范围: {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')
print(f'列: {list(df.columns)}')

In [ ]:
# ============================================================
# Cell 6: GP 表达式树定义与核心算法
# ============================================================

# --- 原语 (叶节点) ---
PRIMITIVES = ['close', 'open', 'high', 'low', 'volume']

# --- 运算符 (内部节点) ---
# 一元运算: rank
# 二元运算: +, -, *, /
# 时序运算: ts_mean(x, d), ts_std(x, d), delay(x, d)
UNARY_OPS = ['rank']
BINARY_OPS = ['add', 'sub', 'mul', 'div']
TS_OPS = ['ts_mean', 'ts_std', 'delay']  # 带窗口参数
TS_WINDOWS = [3, 5, 10, 20]


class Node:
    """表达式树节点"""
    def __init__(self, op=None, children=None, value=None, window=None):
        self.op = op              # 运算符名
        self.children = children or []  # 子节点列表
        self.value = value        # 叶节点的原语名称
        self.window = window      # 时序运算的窗口参数

    def is_leaf(self):
        return self.op is None

    def depth(self):
        if self.is_leaf():
            return 1
        return 1 + max(c.depth() for c in self.children)

    def __str__(self):
        if self.is_leaf():
            return self.value
        if self.op in UNARY_OPS:
            return f'{self.op}({self.children[0]})'
        if self.op in BINARY_OPS:
            op_sym = {'+': '+', 'add': '+', 'sub': '-', 'mul': '*', 'div': '/'}[self.op]
            return f'({self.children[0]} {op_sym} {self.children[1]})'
        if self.op in TS_OPS:
            return f'{self.op}({self.children[0]}, {self.window})'
        return '?'

    def evaluate(self, data):
        """在OHLCV数据上求值, data是DataFrame含close/open/high/low/volume"""
        if self.is_leaf():
            return data[self.value].astype(float).copy()
        if self.op == 'rank':
            child_val = self.children[0].evaluate(data)
            return child_val.rank(pct=True)
        if self.op == 'add':
            return self.children[0].evaluate(data) + self.children[1].evaluate(data)
        if self.op == 'sub':
            return self.children[0].evaluate(data) - self.children[1].evaluate(data)
        if self.op == 'mul':
            a = self.children[0].evaluate(data)
            b = self.children[1].evaluate(data)
            return a * b
        if self.op == 'div':
            a = self.children[0].evaluate(data)
            b = self.children[1].evaluate(data)
            b = b.replace(0, np.nan)
            return a / b
        if self.op == 'ts_mean':
            child_val = self.children[0].evaluate(data)
            return child_val.rolling(self.window, min_periods=1).mean()
        if self.op == 'ts_std':
            child_val = self.children[0].evaluate(data)
            return child_val.rolling(self.window, min_periods=2).std().fillna(0)
        if self.op == 'delay':
            child_val = self.children[0].evaluate(data)
            return child_val.shift(self.window).fillna(method='bfill')
        return pd.Series(0, index=data.index)


def random_tree(max_depth=5, current_depth=0):
    """生成随机表达式树"""
    if current_depth >= max_depth or (current_depth > 0 and random.random() < 0.3):
        return Node(value=random.choice(PRIMITIVES))

    r = random.random()
    if r < 0.15:  # 一元运算
        op = random.choice(UNARY_OPS)
        child = random_tree(max_depth, current_depth + 1)
        return Node(op=op, children=[child])
    elif r < 0.55:  # 二元运算
        op = random.choice(BINARY_OPS)
        left = random_tree(max_depth, current_depth + 1)
        right = random_tree(max_depth, current_depth + 1)
        return Node(op=op, children=[left, right])
    else:  # 时序运算
        op = random.choice(TS_OPS)
        child = random_tree(max_depth, current_depth + 1)
        window = random.choice(TS_WINDOWS)
        return Node(op=op, children=[child], window=window)


def get_all_nodes(tree):
    """获取树中所有节点的列表"""
    nodes = [tree]
    for c in tree.children:
        nodes.extend(get_all_nodes(c))
    return nodes


def crossover(tree1, tree2):
    """交叉: 交换两棵树的随机子树"""
    t1 = copy.deepcopy(tree1)
    t2 = copy.deepcopy(tree2)
    nodes1 = get_all_nodes(t1)
    nodes2 = get_all_nodes(t2)
    if len(nodes1) < 2 or len(nodes2) < 2:
        return t1, t2
    # 选择交叉点 (排除根节点以保证不同)
    n1 = random.choice(nodes1[1:]) if len(nodes1) > 1 else nodes1[0]
    n2 = random.choice(nodes2[1:]) if len(nodes2) > 1 else nodes2[0]
    # 交换属性
    n1.op, n2.op = n2.op, n1.op
    n1.children, n2.children = n2.children, n1.children
    n1.value, n2.value = n2.value, n1.value
    n1.window, n2.window = n2.window, n1.window
    # 限制深度
    if t1.depth() > 6:
        t1 = random_tree(max_depth=5)
    if t2.depth() > 6:
        t2 = random_tree(max_depth=5)
    return t1, t2


def mutate(tree, max_depth=5):
    """变异: 随机替换子树"""
    t = copy.deepcopy(tree)
    nodes = get_all_nodes(t)
    if len(nodes) < 1:
        return random_tree(max_depth)
    target = random.choice(nodes)
    new_sub = random_tree(max_depth=3)
    target.op = new_sub.op
    target.children = new_sub.children
    target.value = new_sub.value
    target.window = new_sub.window
    if t.depth() > 6:
        return random_tree(max_depth)
    return t


def compute_ic(alpha_values, forward_returns):
    """计算 IC (Spearman 秩相关)"""
    valid = pd.DataFrame({'alpha': alpha_values, 'ret': forward_returns}).dropna()
    if len(valid) < 30:
        return 0.0
    ic, _ = stats.spearmanr(valid['alpha'], valid['ret'])
    return ic if not np.isnan(ic) else 0.0


def evaluate_fitness(tree, data, forward_returns):
    """评估表达式树的适应度 (IC)"""
    try:
        alpha = tree.evaluate(data)
        if alpha.isna().all() or alpha.std() == 0:
            return -1.0
        ic = compute_ic(alpha, forward_returns)
        return abs(ic)  # 取绝对值, 负IC也是有效信号
    except Exception:
        return -1.0


print('GP 核心组件定义完成')
print(f'原语: {PRIMITIVES}')
print(f'运算符: {UNARY_OPS + BINARY_OPS + TS_OPS}')

# 快速测试
test_tree = random_tree(max_depth=4)
print(f'随机表达式示例: {test_tree}')
print(f'树深度: {test_tree.depth()}')

In [ ]:
# ============================================================
# Cell 7: 遗传编程进化主循环
# ============================================================

# 计算未来5日收益率作为标签
forward_ret = df['close'].pct_change(5).shift(-5)

# GP 参数
POP_SIZE = 50
N_GENERATIONS = 30
MAX_DEPTH = 5
TOURNAMENT_SIZE = 5
CROSSOVER_PROB = 0.7
MUTATION_PROB = 0.2
ELITE_SIZE = 5

# 初始化种群
population = [random_tree(MAX_DEPTH) for _ in range(POP_SIZE)]

# 进化记录
history = {
    'generation': [],
    'best_ic': [],
    'avg_ic': [],
    'best_expr': [],
    'avg_depth': [],
}

print(f'开始进化: population={POP_SIZE}, generations={N_GENERATIONS}')
print('=' * 60)

for gen in range(N_GENERATIONS):
    # 评估适应度
    fitnesses = [evaluate_fitness(ind, df, forward_ret) for ind in population]

    # 记录统计
    valid_fits = [f for f in fitnesses if f >= 0]
    best_idx = np.argmax(fitnesses)
    best_fit = fitnesses[best_idx]
    avg_fit = np.mean(valid_fits) if valid_fits else 0
    avg_depth = np.mean([ind.depth() for ind in population])

    history['generation'].append(gen)
    history['best_ic'].append(best_fit)
    history['avg_ic'].append(avg_fit)
    history['best_expr'].append(str(population[best_idx]))
    history['avg_depth'].append(avg_depth)

    if gen % 5 == 0:
        print(f'Gen {gen:3d} | Best IC: {best_fit:.4f} | Avg IC: {avg_fit:.4f} | '
              f'Avg Depth: {avg_depth:.1f} | Best: {str(population[best_idx])[:60]}')

    # 精英保留
    sorted_indices = np.argsort(fitnesses)[::-1]
    elites = [copy.deepcopy(population[i]) for i in sorted_indices[:ELITE_SIZE]]

    # 锦标赛选择 + 交叉 + 变异
    new_pop = list(elites)

    while len(new_pop) < POP_SIZE:
        # 锦标赛选择两个父代
        candidates = random.sample(range(POP_SIZE), TOURNAMENT_SIZE)
        parent1_idx = max(candidates, key=lambda i: fitnesses[i])
        candidates = random.sample(range(POP_SIZE), TOURNAMENT_SIZE)
        parent2_idx = max(candidates, key=lambda i: fitnesses[i])

        p1 = copy.deepcopy(population[parent1_idx])
        p2 = copy.deepcopy(population[parent2_idx])

        # 交叉
        if random.random() < CROSSOVER_PROB:
            p1, p2 = crossover(p1, p2)

        # 变异
        if random.random() < MUTATION_PROB:
            p1 = mutate(p1, MAX_DEPTH)
        if random.random() < MUTATION_PROB:
            p2 = mutate(p2, MAX_DEPTH)

        new_pop.append(p1)
        if len(new_pop) < POP_SIZE:
            new_pop.append(p2)

    population = new_pop[:POP_SIZE]

# 最终评估
final_fitnesses = [evaluate_fitness(ind, df, forward_ret) for ind in population]
best_idx = np.argmax(final_fitnesses)
best_alpha_tree = population[best_idx]
best_ic = final_fitnesses[best_idx]

print('=' * 60)
print(f'\n进化完成!')
print(f'最佳表达式: {best_alpha_tree}')
print(f'最佳 IC: {best_ic:.4f}')
print(f'表达式深度: {best_alpha_tree.depth()}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

# 用最佳Alpha因子生成交易信号
alpha_signal = best_alpha_tree.evaluate(df)
alpha_signal = alpha_signal.dropna()

# 标准化 alpha 信号
alpha_z = (alpha_signal - alpha_signal.rolling(60).mean()) / alpha_signal.rolling(60).std()
alpha_z = alpha_z.dropna()

# 信号规则: alpha_z > 1 做多, alpha_z < -1 做空, 否则空仓
signals = pd.Series(0, index=alpha_z.index)
signals[alpha_z > 0.5] = 1
signals[alpha_z < -0.5] = -1

# 回测
backtester = Backtester(initial_capital=1_000_000, t_plus_1=True)

# 获取基准
try:
    bench = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
    bench_prices = bench['close']
except Exception:
    bench_prices = None

result = backtester.run(
    prices=df['close'],
    signals=signals,
    benchmark_prices=bench_prices
)

print('=== AutoAlpha GP 策略回测结果 ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 收益曲线
benchmark_equity = None
if result.get('benchmark_returns') is not None:
    benchmark_equity = (1 + result['benchmark_returns']).cumprod() * result['equity_curve'].iloc[0]
plot_equity_curve(result['equity_curve'], benchmark_equity,
                  title='AutoAlpha GP - 累计收益')

# 2. 回撤
plot_drawdown(result['equity_curve'], title='AutoAlpha GP - 回撤')

# 3. 进化曲线 (matplotlib)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['generation'], history['best_ic'], 'r-o', markersize=3, label='最佳 IC')
ax1.plot(history['generation'], history['avg_ic'], 'b--', alpha=0.7, label='平均 IC')
ax1.set_xlabel('代数')
ax1.set_ylabel('IC (绝对值)')
ax1.set_title('GP 进化收敛曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(history['generation'], history['avg_depth'], color='#4CAF50', alpha=0.6)
ax2.set_xlabel('代数')
ax2.set_ylabel('平均树深度')
ax2.set_title('表达式复杂度变化')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. 绩效表
plot_metrics_table(result['metrics'], title='AutoAlpha GP 绩效指标')

## 结果讨论

### 策略表现

1. **表达式进化**: GP 在 30 代内通常能找到 IC 绝对值 > 0.03 的 Alpha 因子
2. **因子多样性**: 不同运行会产生不同结构的表达式树，体现了搜索空间的丰富性
3. **树深度控制**: max_depth=5 限制了表达式复杂度，防止过拟合

### 局限性

- 单资产 (ETF) 回测简化了实际多股票截面选股场景
- IC 作为适应度未考虑因子的换手率和交易成本
- 小种群 (50) 和少代数 (30) 限制了搜索充分性
- 表达式树的数值稳定性需要更多保护措施

### 改进方向

- 加入 ICIR (IC 的均值/标准差) 作为多目标适应度
- 实现层次化进化: 先进化简单子表达式，再组合
- 加入因子正交化约束，避免生成高度相关的因子
- 扩展到多股票截面 IC 评估